In [1]:
import json
import os
from pathlib import Path
import pandas as pd
from collections import Counter

# Define file paths
BASE_DIR = Path(r"E:\Python Projects\SwifTBaSkeT-AI\data")
FILES = {
    "products": BASE_DIR / "products.jsonl",
    "orders": BASE_DIR / "orders.jsonl",
    "returns": BASE_DIR / "returns.jsonl"
}

EXPECTED_COUNTS = {
    "products": 2314,
    "orders": 10000,
    "returns": 28866
}

# Global state for loaded documents and validation tracking
loaded_corpora = {"products": [], "orders": [], "returns": []}
validation_summary = {
    corpus: {
        "Expected Docs": EXPECTED_COUNTS[corpus],
        "Actual Docs": 0,
        "Unique IDs": 0,
        "Invalid JSON": 0,
        "Schema Errors": 0,
        "Metadata Errors": 0,
        "ID Errors": 0,
        "Content Warnings": 0,
        "Status": "PENDING"
    }
    for corpus in FILES
}

print("Initialization complete. Paths defined.")

Initialization complete. Paths defined.


In [2]:
print("SECTION 1 — FILE EXISTENCE AND BASIC FILE VALIDATION\n" + "="*50)

for corpus, filepath in FILES.items():
    print(f"\nChecking corpus: {corpus.upper()}")
    
    # 1. Existence and Size
    if not filepath.exists():
        print(f"  [ERROR] File not found: {filepath}")
        validation_summary[corpus]["Status"] = "FILE_MISSING"
        continue
        
    file_size_mb = filepath.stat().st_size / (1024 * 1024)
    print(f"  File Path: {filepath}")
    print(f"  File Size: {file_size_mb:.2f} MB")
    
    # 2. Line counting and JSON parsing
    valid_docs = []
    invalid_json_count = 0
    empty_doc_count = 0
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                empty_doc_count += 1
                continue
                
            try:
                doc = json.loads(line)
                valid_docs.append(doc)
            except json.JSONDecodeError:
                invalid_json_count += 1
                if invalid_json_count <= 3: # Print first few malformed lines
                    print(f"  [ERROR] Malformed JSON on line {line_num}: {line[:50]}...")
    
    # Update state
    loaded_corpora[corpus] = valid_docs
    validation_summary[corpus]["Actual Docs"] = len(valid_docs)
    validation_summary[corpus]["Invalid JSON"] = invalid_json_count
    
    print(f"  Total Lines Parsed (Valid JSON): {len(valid_docs)}")
    if invalid_json_count > 0:
        print(f"  [ERROR] Invalid JSON lines found: {invalid_json_count}")
    if empty_doc_count > 0:
        print(f"  [WARNING] Empty lines found: {empty_doc_count}")

SECTION 1 — FILE EXISTENCE AND BASIC FILE VALIDATION

Checking corpus: PRODUCTS
  File Path: E:\Python Projects\SwifTBaSkeT-AI\data\products.jsonl
  File Size: 1.68 MB
  Total Lines Parsed (Valid JSON): 2314

Checking corpus: ORDERS
  File Path: E:\Python Projects\SwifTBaSkeT-AI\data\orders.jsonl
  File Size: 12.65 MB
  Total Lines Parsed (Valid JSON): 10000

Checking corpus: RETURNS
  File Path: E:\Python Projects\SwifTBaSkeT-AI\data\returns.jsonl
  File Size: 28.00 MB
  Total Lines Parsed (Valid JSON): 28866


In [3]:
print("SECTION 2 — DOCUMENT SCHEMA VALIDATION\n" + "="*50)

EXPECTED_DOC_TYPES = {
    "products": "product",
    "orders": "order",
    "returns": "return"
}

for corpus, docs in loaded_corpora.items():
    schema_errors = 0
    expected_doc_type = EXPECTED_DOC_TYPES[corpus]
    
    for idx, doc in enumerate(docs):
        doc_id = doc.get("id", f"Row-{idx}")
        errors = []
        
        # Check basic schema keys
        if "id" not in doc or not str(doc["id"]).strip():
            errors.append("Missing or empty 'id'")
        if "text" not in doc or not str(doc["text"]).strip():
            errors.append("Missing or empty 'text'")
        if "metadata" not in doc or not isinstance(doc["metadata"], dict):
            errors.append("Missing or invalid 'metadata' dictionary")
        else:
            doc_type = doc["metadata"].get("doc_type")
            if not doc_type:
                errors.append("Missing 'doc_type' in metadata")
            elif doc_type != expected_doc_type:
                errors.append(f"Invalid doc_type. Expected '{expected_doc_type}', got '{doc_type}'")
                
        if errors:
            schema_errors += 1
            if schema_errors <= 3:
                print(f"  [ERROR] Corpus '{corpus}', Doc '{doc_id}': {', '.join(errors)}")
                
    validation_summary[corpus]["Schema Errors"] = schema_errors
    print(f"{corpus.upper()}: {schema_errors} schema errors detected.")

SECTION 2 — DOCUMENT SCHEMA VALIDATION
PRODUCTS: 0 schema errors detected.
ORDERS: 0 schema errors detected.
RETURNS: 0 schema errors detected.


In [4]:
print("SECTION 3 — DOCUMENT ID VALIDATION\n" + "="*50)

for corpus, docs in loaded_corpora.items():
    all_ids = [str(doc.get("id")) for doc in docs if doc.get("id")]
    unique_ids = set(all_ids)
    
    duplicate_count = len(all_ids) - len(unique_ids)
    validation_summary[corpus]["Unique IDs"] = len(unique_ids)
    validation_summary[corpus]["ID Errors"] = duplicate_count
    
    print(f"\nCorpus: {corpus.upper()}")
    print(f"  Total Documents: {len(all_ids)}")
    print(f"  Unique IDs:      {len(unique_ids)}")
    
    if duplicate_count > 0:
        print(f"  [ERROR] {duplicate_count} duplicate IDs found.")
        id_counts = Counter(all_ids)
        duplicates = [doc_id for doc_id, count in id_counts.items() if count > 1]
        print(f"  Sample duplicates: {duplicates[:3]}")
    else:
        print("  All IDs are unique. [PASS]")

SECTION 3 — DOCUMENT ID VALIDATION

Corpus: PRODUCTS
  Total Documents: 2314
  Unique IDs:      2314
  All IDs are unique. [PASS]

Corpus: ORDERS
  Total Documents: 10000
  Unique IDs:      10000
  All IDs are unique. [PASS]

Corpus: RETURNS
  Total Documents: 28866
  Unique IDs:      28866
  All IDs are unique. [PASS]


In [5]:
print("SECTION 4 — METADATA QUALITY VALIDATION\n" + "="*50)

REQUIRED_METADATA = {
    "products": ["doc_type", "product_id", "sku", "category", "sub_category", "brand", "brand_tier", "product_status", "launch_status", "is_perishable"],
    "orders": ["doc_type", "order_id", "customer_id", "store_id", "rider_id", "order_status", "payment_method", "payment_status", "order_source"],
    "returns": ["doc_type", "return_id", "order_id", "product_id", "customer_id", "return_status", "return_reason", "brand", "category"]
}

for corpus, docs in loaded_corpora.items():
    req_fields = REQUIRED_METADATA[corpus]
    metadata_errors = 0
    missing_fields_counter = Counter()
    
    for doc in docs:
        meta = doc.get("metadata", {})
        doc_id = doc.get("id", "Unknown")
        missing = [field for field in req_fields if field not in meta or meta[field] is None]
        
        if missing:
            metadata_errors += 1
            missing_fields_counter.update(missing)
            if metadata_errors <= 3:
                 print(f"  [ERROR] Corpus '{corpus}', Doc '{doc_id}' missing metadata: {missing}")
                 
    validation_summary[corpus]["Metadata Errors"] = metadata_errors
    print(f"{corpus.upper()}: {metadata_errors} documents with metadata errors.")
    if missing_fields_counter:
        print(f"  Most common missing fields: {missing_fields_counter.most_common(3)}")

SECTION 4 — METADATA QUALITY VALIDATION
PRODUCTS: 0 documents with metadata errors.
ORDERS: 0 documents with metadata errors.
RETURNS: 0 documents with metadata errors.


In [6]:
print("SECTION 5 — CROSS-CORPUS REFERENTIAL VALIDATION\n" + "="*50)

# Extract reference sets
product_ids = {str(doc.get("id")) for doc in loaded_corpora["products"]}
order_ids = {str(doc.get("id")) for doc in loaded_corpora["orders"]}

# 1. Product IDs inside Orders
print("Validation 1: Products referenced in Orders")
print("  [INFO] Due to the document generation structure, order documents aggregate items into a text/JSON block inside the 'text' field. "
      "Extracting 'product_id' reliably using regex from unstructured text is fragile. "
      "Since 'product_id' was deliberately kept out of order metadata to prevent bloating, "
      "this specific cross-check cannot be performed programmatically from the current corpus format. Proceeding to direct references...")

# 2. Order IDs inside Returns
print("\nValidation 2: Orders referenced in Returns")
returns_orphan_orders = 0
for doc in loaded_corpora["returns"]:
    ref_order = str(doc.get("metadata", {}).get("order_id"))
    if ref_order and ref_order not in order_ids:
        # Note: Depending on the 10,000 order limit set in MVP, older returns might reference orders not in the 10K subset.
        returns_orphan_orders += 1

print(f"  Found {returns_orphan_orders} Return documents referencing missing Order IDs.")
if returns_orphan_orders > 0:
    print("  [WARNING] Expected behavior if returns span 300K orders but the MVP orders.jsonl only contains the 10,000 most recent.")

# 3. Product IDs inside Returns
print("\nValidation 3: Products referenced in Returns")
returns_orphan_products = 0
for doc in loaded_corpora["returns"]:
    ref_prod = str(doc.get("metadata", {}).get("product_id"))
    if ref_prod and ref_prod not in product_ids:
        returns_orphan_products += 1

print(f"  Found {returns_orphan_products} Return documents referencing missing Product IDs.")
if returns_orphan_products > 0:
    print("  [ERROR] Referential integrity failure for products in returns.")

SECTION 5 — CROSS-CORPUS REFERENTIAL VALIDATION
Validation 1: Products referenced in Orders
  [INFO] Due to the document generation structure, order documents aggregate items into a text/JSON block inside the 'text' field. Extracting 'product_id' reliably using regex from unstructured text is fragile. Since 'product_id' was deliberately kept out of order metadata to prevent bloating, this specific cross-check cannot be performed programmatically from the current corpus format. Proceeding to direct references...

Validation 2: Orders referenced in Returns
  Found 27822 Return documents referencing missing Order IDs.
  [WARNING] Expected behavior if returns span 300K orders but the MVP orders.jsonl only contains the 10,000 most recent.

Validation 3: Products referenced in Returns
  Found 0 Return documents referencing missing Product IDs.


In [7]:
print("SECTION 6 — CONTENT QUALITY CHECKS\n" + "="*50)

for corpus, docs in loaded_corpora.items():
    if not docs:
        continue
        
    lengths = [len(str(doc.get("text", ""))) for doc in docs]
    avg_len = sum(lengths) / len(lengths)
    short_docs = sum(1 for l in lengths if l < 50)
    na_heavy_docs = sum(1 for doc in docs if str(doc.get("text", "")).count("N/A") > 5)
    
    # ID matching check
    id_mismatches = 0
    id_field_map = {"products": "product_id", "orders": "order_id", "returns": "return_id"}
    target_meta_id = id_field_map[corpus]
    
    for doc in docs:
        if str(doc.get("id")) != str(doc.get("metadata", {}).get(target_meta_id)):
            id_mismatches += 1
            
    validation_summary[corpus]["Content Warnings"] = short_docs + na_heavy_docs + id_mismatches

    print(f"\nCorpus: {corpus.upper()}")
    print(f"  Text Length - Min: {min(lengths)}, Max: {max(lengths)}, Avg: {avg_len:.0f}")
    print(f"  Unusually short docs (< 50 chars): {short_docs}")
    print(f"  Docs with > 5 'N/A' values: {na_heavy_docs}")
    print(f"  Top-level ID vs Metadata ID mismatches: {id_mismatches}")

SECTION 6 — CONTENT QUALITY CHECKS

Corpus: PRODUCTS
  Text Length - Min: 412, Max: 503, Avg: 452
  Unusually short docs (< 50 chars): 0
  Docs with > 5 'N/A' values: 0
  Top-level ID vs Metadata ID mismatches: 0

Corpus: ORDERS
  Text Length - Min: 646, Max: 3834, Avg: 1044
  Unusually short docs (< 50 chars): 0
  Docs with > 5 'N/A' values: 0
  Top-level ID vs Metadata ID mismatches: 0

Corpus: RETURNS
  Text Length - Min: 661, Max: 769, Avg: 705
  Unusually short docs (< 50 chars): 0
  Docs with > 5 'N/A' values: 0
  Top-level ID vs Metadata ID mismatches: 0


In [8]:
print("SECTION 7 — SEARCHABILITY / RETRIEVAL SANITY CHECK\n" + "="*50)

SANITY_QUERIES = [
    ("cheese balls", ["products", "orders"]),
    ("mccain", ["products"]),
    ("upi", ["orders"]),
    ("damaged", ["returns"]),
    ("approved", ["returns"]),
    ("snacks", ["products", "returns"])
]

for keyword, target_corpora in SANITY_QUERIES:
    print(f"\nSearch Term: '{keyword}'")
    for corpus in target_corpora:
        matches = [doc for doc in loaded_corpora[corpus] if keyword in str(doc.get("text", "")).lower()]
        print(f"  Found in {corpus.upper()}: {len(matches)} documents")
        
        if matches:
            print("  Sample Matches:")
            for m in matches[:2]:
                text_preview = m['text'].replace('\n', ' ')[:100] + "..."
                print(f"    - [{m['id']}] {text_preview}")

SECTION 7 — SEARCHABILITY / RETRIEVAL SANITY CHECK

Search Term: 'cheese balls'
  Found in PRODUCTS: 34 documents
  Sample Matches:
    - [P000001] Product Name: McCain Fries & Bites Cheese Balls Veg Spicy 750.0 g. Brand: McCain (Tier: Premium). Ca...
    - [P000023] Product Name: Amul Fries & Bites Cheese Balls Chicken Masala 750.0 g. Brand: Amul (Tier: Popular). C...
  Found in ORDERS: 178 documents
  Sample Matches:
    - [ORD0299988] Order ID: ORD0299988 Timestamp: 2025-09-30 23:49:37 Order Status: Delivered. Delivery Details: Slot ...
    - [ORD0299976] Order ID: ORD0299976 Timestamp: 2025-09-30 23:32:14 Order Status: Delivered. Delivery Details: Slot ...

Search Term: 'mccain'
  Found in PRODUCTS: 42 documents
  Sample Matches:
    - [P000001] Product Name: McCain Fries & Bites Cheese Balls Veg Spicy 750.0 g. Brand: McCain (Tier: Premium). Ca...
    - [P000018] Product Name: McCain Fries & Bites French Fries Chicken Classic 400.0 g. Brand: McCain (Tier: Premiu...

Search Term: 'u

In [10]:
print("=" * 70)
print("SWIFTBASKET RAG CORPUS — FINAL VALIDATION REPORT")
print("=" * 70)

for corpus, summary in validation_summary.items():
    print(f"\n{corpus.upper()}")
    print("-" * 40)
    
    for key, value in summary.items():
        print(f"{key:<25}: {value}")

# Overall status
critical_failures = []

for corpus, summary in validation_summary.items():
    if summary["Actual Docs"] != summary["Expected Docs"]:
        critical_failures.append(f"{corpus}: document count mismatch")
    if summary["Invalid JSON"] > 0:
        critical_failures.append(f"{corpus}: invalid JSON")
    if summary["Schema Errors"] > 0:
        critical_failures.append(f"{corpus}: schema errors")
    if summary["Metadata Errors"] > 0:
        critical_failures.append(f"{corpus}: metadata errors")
    if summary["ID Errors"] > 0:
        critical_failures.append(f"{corpus}: duplicate IDs")

print("\n" + "=" * 70)

if not critical_failures:
    print("OVERALL STATUS: PASSED")
    print("All critical corpus validation checks passed.")
else:
    print("OVERALL STATUS: FAILED")
    print("\nCritical failures:")
    for failure in critical_failures:
        print(f"  - {failure}")

print("=" * 70)

SWIFTBASKET RAG CORPUS — FINAL VALIDATION REPORT

PRODUCTS
----------------------------------------
Expected Docs            : 2314
Actual Docs              : 2314
Unique IDs               : 2314
Invalid JSON             : 0
Schema Errors            : 0
Metadata Errors          : 0
ID Errors                : 0
Content Warnings         : 0
Status                   : PASS

ORDERS
----------------------------------------
Expected Docs            : 10000
Actual Docs              : 10000
Unique IDs               : 10000
Invalid JSON             : 0
Schema Errors            : 0
Metadata Errors          : 0
ID Errors                : 0
Content Warnings         : 0
Status                   : PASS

RETURNS
----------------------------------------
Expected Docs            : 28866
Actual Docs              : 28866
Unique IDs               : 28866
Invalid JSON             : 0
Schema Errors            : 0
Metadata Errors          : 0
ID Errors                : 0
Content Warnings         : 0
Status  